# 01 — Mineração multi-dia de erros de improdutividade

Este caderno usa `correcoes.csv` somente como **challenge set**. Como todas as linhas já são correções, ele revela padrões e contraprovas, mas não possui o denominador necessário para estimar precisão, recall ou coverage da população.

Regras de integridade: a verdade é `label_corrigido`/`cat_corr`; nenhuma candidata pode usar esses campos como entrada; descrições antigas não são tratadas como verdade; CAM1 e CAM2 são agrupadas pelo token do episódio físico.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

for pasta in (Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks'):
    if (pasta / 'improdutividade_eval.py').is_file():
        sys.path.insert(0, str(pasta))
        break
from improdutividade_eval import I, P, classe_lean, episode_id, localizar_dados, catalogo_candidatas

DATA_ROOT = localizar_dados('correcoes.csv')
print('Dados carregados: correcoes.csv')

Dados carregados: correcoes.csv


In [2]:
df = pd.read_csv(DATA_ROOT / 'correcoes.csv')
df['data'] = pd.to_datetime(df['gravado_em'], utc=True, errors='coerce').dt.date.astype(str)
df['episode_id'] = [episode_id(n, g) for n, g in zip(df['video_nome'], df['gravado_em'])]
df['y_original'] = df['cat_orig'].map(classe_lean)
df['y_corrigido'] = df['cat_corr'].map(classe_lean)
df['dur_s'] = pd.to_numeric(df['dur_s'], errors='coerce').fillna(0).clip(lower=0)

assert df['label_corrigido'].notna().all(), 'Há correção sem rótulo humano.'
assert (df['comportamento_label'] != df['label_corrigido']).all(), 'Não é um conjunto só de erros.'
alvos_proibidos = {'label_corrigido', 'novo', 'cat_corr', 'mudou_cat'}
entradas_candidatas = {'comportamento_label', 'papel_pessoa', 'trabalho', 'orientacao',
                       'maos_maquina', 'modo_operacao', 'movimento_maquina',
                       'cena_maquina', 'cena_imovel', 'versao_instrumento'}
assert entradas_candidatas.isdisjoint(alvos_proibidos), 'Leakage na definição de candidatas.'

resumo = pd.Series({
    'linhas_corrigidas': len(df),
    'episodios_fisicos': df['episode_id'].nunique(),
    'videos_uuid': df['video_id'].nunique(),
    'dias': df['data'].nunique(),
    'duracao_h': df['dur_s'].sum() / 3600,
    'todas_as_linhas_sao_erros': bool((df['comportamento_label'] != df['label_corrigido']).all()),
})
display(resumo.to_frame('valor'))
print('CONCLUSÃO METODOLÓGICA: não calcular precision/recall/coverage neste arquivo.')

,valor
linhas_corrigidas,423
episodios_fisicos,258
videos_uuid,259
dias,19
duracao_h,6.617514
todas_as_linhas_sao_erros,True


CONCLUSÃO METODOLÓGICA: não calcular precision/recall/coverage neste arquivo.


In [3]:
transicoes = (df.groupby(['y_original', 'y_corrigido'])
              .agg(casos=('id', 'size'), minutos=('dur_s', lambda s: s.sum()/60),
                   dias=('data', 'nunique'), episodios=('episode_id', 'nunique'))
              .sort_values(['casos'], ascending=False))
display(transicoes)

falsas_acusacoes = df[(df['y_original'] == I) & (df['y_corrigido'] == P)]
improdutivos_perdidos = df[(df['y_original'] == P) & (df['y_corrigido'] == I)]
print(f'Falsas acusações conhecidas I→P: {len(falsas_acusacoes)}')
print(f'Improdutivos conhecidos perdidos Pâ†’I: {len(improdutivos_perdidos)}')
print(f'Correções sem mudança binária: {(df.y_original == df.y_corrigido).sum()}')

casos     minutos  dias  episodios
y_original  y_corrigido                                    
IMPRODUTIVO PRODUTIVO      172  159.128167    14        108
PRODUTIVO   IMPRODUTIVO    151  137.956000    16         99
            PRODUTIVO       70   69.966667    12         54
IMPRODUTIVO IMPRODUTIVO     30   30.000000     9         24

Falsas acusações conhecidas I→P: 172
Improdutivos conhecidos perdidos Pâ†’I: 151
Correções sem mudança binária: 100


In [4]:
padroes = (df.groupby(['comportamento_label', 'y_original', 'y_corrigido'])
           .agg(casos=('id', 'size'), dias=('data', 'nunique'),
                episodios=('episode_id', 'nunique'), minutos=('dur_s', lambda s: s.sum()/60))
           .reset_index())
padroes = padroes[(padroes['casos'] >= 5) & (padroes['dias'] >= 2)]
display(padroes.sort_values(['casos', 'dias'], ascending=False).head(20))
print('Leitura: recorrência prioriza investigação; não é probabilidade populacional.')

,comportamento_label,y_original,y_corrigido,casos,dias,episodios,minutos
1,acao_indefinida,IMPRODUTIVO,PRODUTIVO,118,4,65,110.486500
8,monitorar_maquina,PRODUTIVO,IMPRODUTIVO,95,16,68,86.508500
12,operar_torno,PRODUTIVO,IMPRODUTIVO,55,10,45,51.378000
9,monitorar_maquina,PRODUTIVO,PRODUTIVO,46,8,37,46.000000
15,posto_vazio,IMPRODUTIVO,PRODUTIVO,36,12,31,31.608333
13,operar_torno,PRODUTIVO,PRODUTIVO,22,9,17,21.966667
0,acao_indefinida,IMPRODUTIVO,IMPRODUTIVO,20,5,15,20.000000
5,conversando_colega,IMPRODUTIVO,PRODUTIVO,15,8,12,14.033333
4,conversando_colega,IMPRODUTIVO,IMPRODUTIVO,8,3,7,8.000000


Leitura: recorrência prioriza investigação; não é probabilidade populacional.


In [5]:
sinais = ['trabalho', 'orientacao', 'maos_maquina', 'modo_operacao',
          'movimento_maquina', 'cena_maquina', 'cena_imovel']
cobertura_sinais = pd.DataFrame({
    'preenchido_pct': [100 * df[c].notna().mean() for c in sinais],
    'primeira_versao_com_dado': [df.loc[df[c].notna(), 'versao_instrumento'].min() for c in sinais],
}, index=sinais)
display(cobertura_sinais.round(2))
display(pd.crosstab(df['data'], df['versao_instrumento']))
print('ALERTA: data e versão do instrumento estão confundidas; comparar dias sem estratificar versão atribui causa errada.')

# Contrafactuais permitidos no challenge set: quantos ERROS conhecidos uma
# trava mandaria para abstenção e quantos acertos negativos conhecidos ela
# também sacrificaria. Isto NÃO é precision/recall da população.
claim_i = df['y_original'].eq(I)
fp_conhecido = claim_i & df['y_corrigido'].eq(P)
tp_conhecido = claim_i & df['y_corrigido'].eq(I)
regras = {
    'C1_abster_acao_indefinida_ou_posto_vazio': df['comportamento_label'].isin(['acao_indefinida','posto_vazio']),
    'C3_veto_sinal_positivo': (df['maos_maquina'].eq(True) | df['modo_operacao'].isin(['manual','automatico']) |
                                df['cena_maquina'].eq('ciclo') | df['movimento_maquina'].eq('intermitente')),
}
regras['C1_C3_combinadas'] = regras['C1_abster_acao_indefinida_ou_posto_vazio'] | regras['C3_veto_sinal_positivo']
linhas = []
for nome, mascara in regras.items():
    linhas.append({
        'regra': nome,
        'FP_I->P_capturados': int((fp_conhecido & mascara).sum()),
        'captura_FP_conhecido_pct': 100 * (fp_conhecido & mascara).sum() / fp_conhecido.sum(),
        'TP_I_preservados': int((tp_conhecido & ~mascara).sum()),
        'preservacao_TP_conhecido_pct': 100 * (tp_conhecido & ~mascara).sum() / tp_conhecido.sum(),
        'dias_com_FP_capturado': df.loc[fp_conhecido & mascara, 'data'].nunique(),
    })
display(pd.DataFrame(linhas).set_index('regra').round(2))
print('Diagnóstico: os vetos capturam erros recorrentes, mas também sacrificam negativos corretos; precisam do holdout, não de promoção direta.')

,preenchido_pct,primeira_versao_com_dado
trabalho,1.42,9
orientacao,3.07,8
maos_maquina,27.66,6
modo_operacao,43.03,6
movimento_maquina,57.92,4
cena_maquina,32.15,4
cena_imovel,58.63,4


versao_instrumento,1,2,3,4,5,6,7,8,9
data,,,,,,,,,
2026-07-27,1,0,0,0,0,0,0,0,0
2026-07-28,28,0,0,0,0,0,0,0,0
2026-07-29,8,0,0,0,0,0,0,0,0
2026-07-30,12,0,0,0,0,0,0,0,0
2026-07-31,18,0,0,0,0,0,0,0,0
2026-08-03,16,0,0,0,0,0,0,0,0
2026-08-04,14,0,0,0,0,0,0,0,0
2026-08-05,38,0,0,0,0,0,0,0,0
2026-08-06,0,4,21,0,0,0,0,0,0


ALERTA: data e versão do instrumento estão confundidas; comparar dias sem estratificar versão atribui causa errada.


,FP_I->P_capturados,captura_FP_conhecido_pct,TP_I_preservados,preservacao_TP_conhecido_pct,dias_com_FP_capturado
regra,,,,,
C1_abster_acao_indefinida_ou_posto_vazio,154,89.53,9,30.00,12
C3_veto_sinal_positivo,84,48.84,16,53.33,4
C1_C3_combinadas,156,90.70,7,23.33,12


Diagnóstico: os vetos capturam erros recorrentes, mas também sacrificam negativos corretos; precisam do holdout, não de promoção direta.


In [6]:
display(catalogo_candidatas())
print('Saída do notebook: famílias de regras pré-registradas para o screening; nenhuma foi declarada vencedora aqui.')

,regra,evidencia_necessaria,status_historico
id,,,
C0,baseline vigente,controle,estimável
C1,separar presença/identidade de atividade,nivel/origem,estimável parcialmente
C2,motivo negativo em whitelist,produtividade_motivo,não estimável no histórico
C3,veto por mãos/máquina ativa,mãos+movimento+modo,não estimável end-to-end
C4,persistência 2-de-3 do mesmo motivo,quadros+track+motivo,não estimável no agregado
C5,conversa só com interlocutor confirmado,bbox_stats.interlocutor,não estimável no CSV
C6,resgate de monitoramento/ponte por episódio,movimento+CAM2+janela,não estimável sem GT


Saída do notebook: famílias de regras pré-registradas para o screening; nenhuma foi declarada vencedora aqui.
